# SQL Practice — Home Credit Default Risk (DuckDB)

A full practice worksheet covering the SQL you need for portfolio and credit-risk analyst work, built on the Home Credit relational tables. Written for **DuckDB**, which you can run two ways:

- **In this notebook** — every section is runnable. A tiny `q(...)` helper sends SQL straight to DuckDB and returns a pandas DataFrame, so you get instant feedback.
- **In DBeaver** — the SQL is standard enough to paste into a DBeaver SQL editor connected to DuckDB (or, with minor tweaks, Postgres). Load each CSV as a table named exactly as the views below (`application_train`, `bureau`, ...), and the queries run unchanged.

**How to use this notebook**

1. Work top to bottom. Each section gives you context and the key syntax, shows **one worked query you can run**, then hands you exercises to write yourself in the empty cells.
2. Every section has **interpretation questions**. Answer them in the markdown cells. In credit risk, saying what a result *means for the portfolio* matters as much as getting the query to run.
3. When a query is heavy (window functions over installments, the capstone feature build), it will be slow the first time — DuckDB is reading a multi-hundred-MB CSV. That is normal.
4. Bring your finished worksheet back for marking — queries and written answers both.

**The tables** (grain = what one row means)

| view | grain | key(s) |
|---|---|---|
| `application_train` | one **current application** (the thing you predict) | `SK_ID_CURR`, has `TARGET` |
| `bureau` | one **external credit** reported to the credit bureau | `SK_ID_BUREAU`, links via `SK_ID_CURR` |
| `bureau_balance` | one **month** of one bureau credit | `SK_ID_BUREAU`, `MONTHS_BALANCE` |
| `previous_application` | one **prior Home Credit application** | `SK_ID_PREV`, links via `SK_ID_CURR` |
| `installments_payments` | one **installment payment** on a prior loan | `SK_ID_PREV` / `SK_ID_CURR` |
| `pos_cash_balance` | one **month** of a prior POS/cash loan | `SK_ID_PREV` / `SK_ID_CURR` |
| `credit_card_balance` | one **month** of a prior credit card | `SK_ID_PREV` / `SK_ID_CURR` |

`TARGET = 1` means the client had payment difficulties (defaulted). Portfolio default rate is therefore just `AVG(TARGET)`. The `DAYS_*` columns are **negative** integer day-counts relative to the application date.

## 0 — Setup and connection

Run the next cell once. It opens an in-memory DuckDB connection, registers each CSV as a **view** (so nothing is copied — DuckDB reads the file on demand), and defines `q(sql)` which returns the result as a DataFrame.

If your CSVs live elsewhere, edit `DATA`. In **DBeaver**, skip this cell — instead import each CSV as a table with the same lowercase name.

In [5]:
import os
import duckdb
import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

DATA = "../../Datasets/Home Credit Default Risk"   # adjust if needed
PARQUET = os.path.join(DATA, "parquet")
os.makedirs(PARQUET, exist_ok=True)

con = duckdb.connect()   # in-memory

TABLES = {
    "application_train":     "application_train.csv",
    "bureau":                "bureau.csv",
    "bureau_balance":        "bureau_balance.csv",
    "previous_application":  "previous_application.csv",
    "installments_payments": "installments_payments.csv",
    "pos_cash_balance":      "POS_CASH_balance.csv",
    "credit_card_balance":   "credit_card_balance.csv",
}

def q(sql):
    """Run SQL against DuckDB and return a pandas DataFrame."""
    return con.execute(sql).df()

print("preparing tables...")
for view, fname in TABLES.items():
    pq = os.path.join(PARQUET, f"{view}.parquet")
    if not os.path.exists(pq):
        print(f"  converting {fname} -> {view}.parquet ...", end=" ", flush=True)
        con.execute(
            f"COPY (SELECT * FROM read_csv_auto('{DATA}/{fname}')) "
            f"TO '{pq}' (FORMAT PARQUET)"
        )
        print("done")
    con.execute(f"CREATE OR REPLACE VIEW {view} AS SELECT * FROM '{pq}'")

print("\nconnected. row counts:")
for v in TABLES:
    print(f"  {v:24s}", q(f"SELECT COUNT(*) AS n FROM {v}").iloc[0, 0])


preparing tables...

connected. row counts:
  application_train        307511
  bureau                   1716428
  bureau_balance           27299925
  previous_application     1670214
  installments_payments    13605401
  pos_cash_balance         10001358
  credit_card_balance      3840312


In [ ]:
# --- pandas companion: load application_train for sanity checks ---
# Run once, right after the DuckDB setup cell. Lets you cross-check any
# SQL result against pandas, e.g. df['COL'].value_counts(dropna=False)
df = pd.read_csv(r"C:\Dev\CreditRiskLearning\Datasets\Home Credit Default Risk\application_train.csv")
print(df.shape)
df.head()

### The one thing to internalise before you start: **grain**

Every bug in analyst SQL comes back to grain. `application_train` is one row per client. `bureau` is *many* rows per client. If you join them naively and then `AVG(AMT_INCOME_TOTAL)`, you have silently weighted each client by how many bureau credits they have — the number is wrong and nobody will notice for months.

The safe pattern, which this whole notebook builds toward, is: **aggregate the child table down to one-row-per-client first, then join.** Keep saying "one row per what?" out loud.

Run this to feel the fan-out — the same client appears many times in `bureau`:

In [6]:
q('''
SELECT SK_ID_CURR, COUNT(*) AS n_bureau_records
FROM bureau
GROUP BY SK_ID_CURR
ORDER BY n_bureau_records DESC
LIMIT 10
''')


,SK_ID_CURR,n_bureau_records
0,120860,116
1,169704,94
2,318065,78
3,251643,61
4,425396,60
5,295809,59
6,129843,58
7,385133,57
8,177014,56
9,280155,55


## 1 — SELECT, FROM, LIMIT: projection

The skeleton of every query is `SELECT columns FROM table`. `LIMIT` caps the rows returned — always use it while exploring a 300k-row table. You can compute expressions in the `SELECT` list and name them with `AS` (an *alias*).

**Key syntax**

```sql
SELECT col_a,
       col_b,
       col_a / col_b         AS ratio,      -- computed column
       col_a * -1.0 / 365.25 AS years       -- alias with AS
FROM   some_table
LIMIT  10;
```

DuckDB is case-insensitive for keywords and identifiers. Integer / integer does integer division in some engines — multiply by `1.0` (or cast) when you want a real ratio.

**Worked example** — a few columns plus two computed ones:

In [7]:
q('''
SELECT SK_ID_CURR,
       AMT_INCOME_TOTAL,
       AMT_CREDIT,
       AMT_CREDIT / AMT_INCOME_TOTAL AS credit_income_ratio,
       -DAYS_BIRTH / 365.25          AS age_years
FROM   application_train
LIMIT  10
''')


,SK_ID_CURR,AMT_INCOME_TOTAL,AMT_CREDIT,credit_income_ratio,age_years
0,100002,202500.0,406597.5,2.007889,25.902806
1,100003,270000.0,1293502.5,4.790750,45.900068
2,100004,67500.0,135000.0,2.000000,52.145106
3,100006,135000.0,312682.5,2.316167,52.032854
4,100007,121500.0,513000.0,4.222222,54.570842
5,100008,99000.0,490495.5,4.954500,46.381930
6,100009,171000.0,1560726.0,9.127053,37.722108
7,100010,360000.0,1530000.0,4.250000,51.608487
8,100011,112500.0,1019610.0,9.063200,55.028063
9,100012,135000.0,405000.0,3.000000,39.613963


### Exercises

**1.1** — Return `SK_ID_CURR`, `AMT_CREDIT`, `AMT_ANNUITY`, and a computed `annuity_income_ratio` (`AMT_ANNUITY / AMT_INCOME_TOTAL`) for the first 10 rows.

**1.2** — Return `SK_ID_CURR` and a column `years_employed` = `-DAYS_EMPLOYED / 365.25`. Limit 10. (You will see a suspicious value — hold that thought for Section 7.)

**1.3** — Return `SK_ID_CURR`, `AMT_GOODS_PRICE`, `AMT_CREDIT`, and `AMT_CREDIT - AMT_GOODS_PRICE AS credit_over_goods` for 10 rows. This difference is roughly the financed amount above the item's price.

**Interpretation 1.1** — Why alias computed columns? What does the output look like if you don't?
**Interpretation 1.2** — `AMT_CREDIT / AMT_INCOME_TOTAL` — in one sentence, what does a value of 5 mean for a borrower?

In [8]:

# Ex 1.1
q('''
SELECT SK_ID_CURR,
    AMT_INCOME_TOTAL,
    AMT_CREDIT,
    AMT_ANNUITY,
    AMT_ANNUITY / AMT_INCOME_TOTAL AS annuity_income_ratio
FROM application_train
LIMIT 10
    
    
''')


,SK_ID_CURR,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,annuity_income_ratio
0,100002,202500.0,406597.5,24700.5,0.121978
1,100003,270000.0,1293502.5,35698.5,0.132217
2,100004,67500.0,135000.0,6750.0,0.100000
3,100006,135000.0,312682.5,29686.5,0.219900
4,100007,121500.0,513000.0,21865.5,0.179963
5,100008,99000.0,490495.5,27517.5,0.277955
6,100009,171000.0,1560726.0,41301.0,0.241526
7,100010,360000.0,1530000.0,42075.0,0.116875
8,100011,112500.0,1019610.0,33826.5,0.300680
9,100012,135000.0,405000.0,20250.0,0.150000


In [ ]:
# Ex 1.2
q('''
SELECT 
    SK_ID_CURR,
    -DAYS_EMPLOYED / 365.25 AS years_employed
FROM application_train
LIMIT 10
''')


In [12]:
# Ex 1.3
q('''
SELECT 
    SK_ID_CURR,
    AMT_GOODS_PRICE,
    AMT_CREDIT,
    AMT_CREDIT - AMT_GOODS_PRICE AS credit_over_goods
FROM application_train
LIMIT 10


''')


,SK_ID_CURR,AMT_GOODS_PRICE,AMT_CREDIT,credit_over_goods
0,100002,351000.0,406597.5,55597.5
1,100003,1129500.0,1293502.5,164002.5
2,100004,135000.0,135000.0,0.0
3,100006,297000.0,312682.5,15682.5
4,100007,513000.0,513000.0,0.0
5,100008,454500.0,490495.5,35995.5
6,100009,1395000.0,1560726.0,165726.0
7,100010,1530000.0,1530000.0,0.0
8,100011,913500.0,1019610.0,106110.0
9,100012,405000.0,405000.0,0.0


**Your answer (1.1, 1.2):**

**I 1.1**: Cleaner output with alias explaining what the column is

**I 1.2**: Debt to income ratio - 5 to 1 ratio (value = 5) means the borrower is heavily leveraged.

...

## 2 — WHERE: filtering rows

`WHERE` keeps only rows where a condition is true. It runs *before* grouping and aggregation. Combine conditions with `AND` / `OR`, and use parentheses to make precedence explicit.

**Key syntax**

```sql
WHERE AMT_INCOME_TOTAL > 100000
  AND CODE_GENDER = 'F'
  AND NAME_EDUCATION_TYPE IN ('Higher education', 'Academic degree')
  AND AMT_ANNUITY BETWEEN 20000 AND 40000     -- inclusive both ends
  AND OCCUPATION_TYPE IS NOT NULL              -- NULL needs IS / IS NOT
  AND ORGANIZATION_TYPE LIKE 'Business%'       -- % = any run of chars
```

Three traps: `= NULL` never matches (use `IS NULL`); string comparisons are case-sensitive on the value; `BETWEEN` is inclusive.

**Worked example** — high-income women on cash loans:

In [ ]:
q('''
SELECT SK_ID_CURR, CODE_GENDER, AMT_INCOME_TOTAL, NAME_CONTRACT_TYPE
FROM   application_train
WHERE  CODE_GENDER = 'F'
  AND  AMT_INCOME_TOTAL > 300000
  AND  NAME_CONTRACT_TYPE = 'Cash loans'
LIMIT  10
''')


### Exercises

**2.1** — Count applications where `AMT_CREDIT > 1000000`. (Use `SELECT COUNT(*)`.)

**2.2** — Return clients who own a car (`FLAG_OWN_CAR = 'Y'`) **and** own realty (`FLAG_OWN_REALTY = 'Y'`) **and** have more than 2 children. Show `SK_ID_CURR`, `CNT_CHILDREN`, `AMT_INCOME_TOTAL`; limit 20.

**2.3** — Return applications where `OCCUPATION_TYPE` is `NULL`. How many are there? (`COUNT(*)` with the right `WHERE`.)

**2.4** — Return clients whose `NAME_INCOME_TYPE` is either `'Working'` or `'State servant'` **and** whose `NAME_EDUCATION_TYPE` is *not* `'Secondary / secondary special'`. Show 20 rows.

**Interpretation 2.1** — From 2.3: a large share of `OCCUPATION_TYPE` is missing. Before you "fix" it, what should you check about *whether the missingness itself* predicts default?
**Interpretation 2.2** — Why is `WHERE col = NULL` always empty, and what do you use instead?

In [17]:
# Ex 2.1
q('''
SELECT COUNT (*) as total_apps
FROM application_train
WHERE
    AMT_CREDIT > 1000000
LIMIT 10
''')


,total_apps
0,49985


In [24]:
df['OCCUPATION_TYPE'].value_counts(dropna=False)

OCCUPATION_TYPE
NaN                      96391
Laborers                 55186
Sales staff              32102
Core staff               27570
Managers                 21371
Drivers                  18603
High skill tech staff    11380
Accountants               9813
Medicine staff            8537
Security staff            6721
Cooking staff             5946
Cleaning staff            4653
Private service staff     2652
Low-skill Laborers        2093
Waiters/barmen staff      1348
Secretaries               1305
Realty agents              751
HR staff                   563
IT staff                   526
Name: count, dtype: int64

In [20]:
# Ex 2.2
q('''
SELECT
    SK_ID_CURR,
    CNT_CHILDREN,
    AMT_INCOME_TOTAL
FROM application_train
WHERE
    FLAG_OWN_CAR = 'Y'
    AND FLAG_OWN_REALTY = 'Y'
    AND CNT_CHILDREN > 2
LIMIT 10

''')


,SK_ID_CURR,CNT_CHILDREN,AMT_INCOME_TOTAL
0,100110,3,135000.0
1,100454,3,607500.0
2,100505,3,225000.0
3,100920,3,112500.0
4,101241,3,180000.0
5,101402,3,90000.0
6,101699,3,292500.0
7,101821,3,202500.0
8,102734,3,135000.0
9,102769,3,90000.0


In [ ]:
# Ex 2.3
q('''
SELECT COUNT (*)
FROM application_train
WHERE 
    OCCUPATION_TYPE IS NULL



''')


In [ ]:
# Ex 2.4
q('''
SELECT *
FROM application_train
WHERE
    NAME_INCOME_TYPE IN ('Working', 'State servant')
    AND NAME_EDUCATION_TYPE != 'Secondary / secondary special'
LIMIT 20
''')

**Your answer (2.1, 2.2):**

**I 2.1:** Check whether the missingness of occuptation type is for those who are unemployed. 

**I 2.2:** Learnt that I have to use "WHERE col IS NULL" to return missing values.
...

## 3 — ORDER BY, DISTINCT, LIMIT/OFFSET

`ORDER BY` sorts the output (last logical step before `LIMIT`). `DISTINCT` removes duplicate rows. `LIMIT n OFFSET m` pages through results.

**Key syntax**

```sql
SELECT DISTINCT NAME_INCOME_TYPE          -- unique values
FROM   application_train;

SELECT ...
ORDER BY AMT_CREDIT DESC, AMT_ANNUITY ASC -- multi-key, mixed direction
LIMIT 20 OFFSET 20;                        -- rows 21..40
```

`NULL`s sort last by default in DuckDB; use `NULLS FIRST` / `NULLS LAST` to control it.

**Worked example** — the 10 largest loans:

In [ ]:
q('''
SELECT SK_ID_CURR, AMT_CREDIT, AMT_ANNUITY, AMT_GOODS_PRICE
FROM   application_train
ORDER BY AMT_CREDIT DESC
LIMIT  10
''')


### Exercises

**3.1** — List the **distinct** values of `NAME_FAMILY_STATUS`.

**3.2** — List the distinct `(NAME_CONTRACT_TYPE, NAME_INCOME_TYPE)` combinations. How many are there?

**3.3** — Show the 15 clients with the highest `AMT_INCOME_TOTAL`, then use `OFFSET` to show the *next* 15 (ranks 16–30).

**3.4** — Show the 10 clients with the smallest positive `AMT_ANNUITY` (exclude `NULL`s). Sort ascending.

**Interpretation 3.1** — You'll notice a client with an enormous `AMT_INCOME_TOTAL` in 3.3. Is "sort and eyeball the top" a reliable way to find data-entry errors? What is one better check?

In [33]:
# Ex 3.1
q('''
SELECT DISTINCT NAME_FAMILY_STATUS
FROM application_train
''')


,NAME_FAMILY_STATUS
0,Widow
1,Married
2,Separated
3,Single / not married
4,Civil marriage
5,Unknown


In [ ]:
# Ex 3.2
q('''
  
  SELECT DISTINCT 
    NAME_CONTRACT_TYPE,
    NAME_INCOME_TYPE
FROM application_train

''')


,NAME_CONTRACT_TYPE,NAME_INCOME_TYPE
0,Revolving loans,Commercial associate
1,Revolving loans,Working
2,Cash loans,Student
3,Cash loans,Maternity leave
4,Cash loans,Commercial associate
5,Revolving loans,Businessman
6,Revolving loans,Pensioner
7,Cash loans,Pensioner
8,Revolving loans,State servant
9,Revolving loans,Unemployed


In [40]:
# Ex 3.3
q('''
SELECT *
    
FROM application_train
ORDER BY AMT_INCOME_TOTAL DESC
LIMIT 15
OFFSET 15
''')


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,NAME_TYPE_SUITE,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,REGION_POPULATION_RELATIVE,DAYS_BIRTH,DAYS_EMPLOYED,DAYS_REGISTRATION,DAYS_ID_PUBLISH,OWN_CAR_AGE,FLAG_MOBIL,FLAG_EMP_PHONE,FLAG_WORK_PHONE,FLAG_CONT_MOBILE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,...,DEF_30_CNT_SOCIAL_CIRCLE,OBS_60_CNT_SOCIAL_CIRCLE,DEF_60_CNT_SOCIAL_CIRCLE,DAYS_LAST_PHONE_CHANGE,FLAG_DOCUMENT_2,FLAG_DOCUMENT_3,FLAG_DOCUMENT_4,FLAG_DOCUMENT_5,FLAG_DOCUMENT_6,FLAG_DOCUMENT_7,FLAG_DOCUMENT_8,FLAG_DOCUMENT_9,FLAG_DOCUMENT_10,FLAG_DOCUMENT_11,FLAG_DOCUMENT_12,FLAG_DOCUMENT_13,FLAG_DOCUMENT_14,FLAG_DOCUMENT_15,FLAG_DOCUMENT_16,FLAG_DOCUMENT_17,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,387126,1,Cash loans,F,Y,Y,1,3150000.0,900000.0,48825.0,900000.0,Unaccompanied,State servant,Higher education,Civil marriage,House / apartment,0.032561,-13668,-1553,-2331.0,-5943,16.0,1,1,1,1,0,0,Private service staff,3.0,...,0.0,0.0,0.0,-585.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
1,217276,0,Revolving loans,M,Y,Y,0,3150000.0,2250000.0,225000.0,2250000.0,Unaccompanied,Commercial associate,Higher education,Married,House / apartment,0.032561,-13386,-5564,-4028.0,-1031,1.0,1,1,1,1,1,0,NaN,2.0,...,0.0,0.0,0.0,-2.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,225210,0,Cash loans,M,Y,Y,0,2930026.5,900000.0,36657.0,900000.0,Unaccompanied,Commercial associate,Secondary / secondary special,Married,House / apartment,0.020713,-15523,-3285,-1110.0,-28,5.0,1,1,1,1,0,0,Managers,2.0,...,0.0,0.0,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,1.0,4.0
3,399467,0,Revolving loans,M,Y,Y,2,2700000.0,180000.0,9000.0,180000.0,Unaccompanied,Working,Higher education,Married,House / apartment,0.010032,-13781,-5756,-223.0,-5183,22.0,1,1,0,1,0,0,Managers,4.0,...,0.0,0.0,0.0,-562.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,2.0,0.0,0.0,0.0,1.0
4,445335,0,Cash loans,M,Y,Y,0,2475000.0,1125000.0,47794.5,1125000.0,Unaccompanied,Working,Secondary / secondary special,Married,House / apartment,0.018209,-10563,-237,-930.0,-2968,21.0,1,1,0,1,1,0,Drivers,2.0,...,0.0,0.0,0.0,-1138.0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,2.0
5,422344,0,Cash loans,M,Y,Y,0,2250000.0,1506816.0,49927.5,1350000.0,Unaccompanied,State servant,Higher education,Married,House / apartment,0.030755,-14318,-5379,-8431.0,-4881,21.0,1,1,0,1,1,0,Core staff,2.0,...,0.0,3.0,0.0,-473.0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,1.0,0.0,1.0
6,441639,0,Revolving loans,F,N,Y,0,2250000.0,675000.0,33750.0,675000.0,Unaccompanied,Working,Higher education,Married,House / apartment,0.018850,-11691,-690,-130.0,-3776,NaN,1,1,0,1,0,0,NaN,2.0,...,0.0,0.0,0.0,-264.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
7,268905,0,Revolving loans,M,Y,N,0,2250000.0,2250000.0,225000.0,2250000.0,Unaccompanied,Commercial associate,Higher education,Married,House / apartment,0.072508,-14347,-5542,-8423.0,-4984,1.0,1,1,0,1,0,0,Managers,2.0,...,0.0,0.0,0.0,-3.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
8,333985,0,Cash loans,F,Y,N,0,2250000.0,1542645.0,62829.0,1440000.0,Unaccompanied,Working,Higher education,Single / not married,House / apartment,0.046220,-15526,-1700,-9635.0,-4105,5.0,1,1,0,1,0,0,Core staff,1.0,...,0.0,0.0,0.0,-871.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
9,111903,0,Revolving loans,M,N,Y,3,2250000.0,900000.0,45000.0,900000.0,Unaccompanied,Working,Higher education,Married,House / apartment,0.009657,-12921,-3725,-2428.0,-4100,NaN,1,1,0,1,0,0,Managers,5.0,...,0.0,0.0,0.0,-618.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,Na

In [55]:
# Ex 3.4
q('''
  
  SELECT *
  FROM application_train
  WHERE AMT_ANNUITY IS NOT NULL
  AND AMT_ANNUITY > 0
  ORDER BY AMT_ANNUITY ASC
  LIMIT 10

''')


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,NAME_TYPE_SUITE,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,REGION_POPULATION_RELATIVE,DAYS_BIRTH,DAYS_EMPLOYED,DAYS_REGISTRATION,DAYS_ID_PUBLISH,OWN_CAR_AGE,FLAG_MOBIL,FLAG_EMP_PHONE,FLAG_WORK_PHONE,FLAG_CONT_MOBILE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,...,DEF_30_CNT_SOCIAL_CIRCLE,OBS_60_CNT_SOCIAL_CIRCLE,DEF_60_CNT_SOCIAL_CIRCLE,DAYS_LAST_PHONE_CHANGE,FLAG_DOCUMENT_2,FLAG_DOCUMENT_3,FLAG_DOCUMENT_4,FLAG_DOCUMENT_5,FLAG_DOCUMENT_6,FLAG_DOCUMENT_7,FLAG_DOCUMENT_8,FLAG_DOCUMENT_9,FLAG_DOCUMENT_10,FLAG_DOCUMENT_11,FLAG_DOCUMENT_12,FLAG_DOCUMENT_13,FLAG_DOCUMENT_14,FLAG_DOCUMENT_15,FLAG_DOCUMENT_16,FLAG_DOCUMENT_17,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,421207,0,Cash loans,F,N,Y,0,31500.0,45000.0,1615.5,45000.0,Unaccompanied,Pensioner,Secondary / secondary special,Single / not married,House / apartment,0.030755,-23489,365243,-3339.0,-4378,NaN,1,0,0,1,1,0,NaN,1.0,...,0.0,0.0,0.0,-1117.0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,1.0,0.0,1.0,1.0
1,185284,0,Cash loans,M,N,Y,0,94500.0,45000.0,1980.0,45000.0,Unaccompanied,Working,Secondary / secondary special,Widow,House / apartment,0.010966,-16840,-1187,-5788.0,-389,NaN,1,1,1,1,1,0,Laborers,1.0,...,0.0,0.0,0.0,-2529.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,1.0,5.0
2,268596,0,Cash loans,F,N,Y,0,67500.0,45000.0,1980.0,45000.0,Unaccompanied,Pensioner,Lower secondary,Married,House / apartment,0.018801,-22656,365243,-13463.0,-4588,NaN,1,0,0,1,1,0,NaN,2.0,...,0.0,0.0,0.0,-793.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,2.0
3,245237,0,Cash loans,F,Y,Y,0,130500.0,49500.0,1993.5,49500.0,Unaccompanied,Pensioner,Higher education,Civil marriage,House / apartment,0.026392,-21344,365243,-1011.0,-3982,3.0,1,0,0,1,0,0,NaN,2.0,...,2.0,5.0,0.0,-729.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,119887,0,Cash loans,F,N,N,0,90000.0,53910.0,2052.0,45000.0,Unaccompanied,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,0.025164,-18739,-433,-1101.0,-2275,NaN,1,1,1,1,0,0,NaN,1.0,...,0.0,8.0,0.0,-1989.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,2.0,4.0
5,357812,0,Cash loans,F,N,Y,0,225000.0,54000.0,2164.5,54000.0,Unaccompanied,Pensioner,Secondary / secondary special,Married,Municipal apartment,0.072508,-23744,365243,-11466.0,-3929,NaN,1,0,0,1,0,0,NaN,2.0,...,0.0,0.0,0.0,-825.0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
6,363161,0,Cash loans,F,N,N,0,36000.0,45000.0,2164.5,45000.0,Unaccompanied,Pensioner,Secondary / secondary special,Civil marriage,House / apartment,0.030755,-20680,365243,-2694.0,-4112,NaN,1,0,0,1,0,0,NaN,2.0,...,0.0,1.0,0.0,-191.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
7,315613,0,Cash loans,F,N,Y,0,157500.0,56880.0,2173.5,45000.0,Unaccompanied,Pensioner,Secondary / secondary special,Widow,House / apartment,0.030755,-21803,365243,-7090.0,-4145,NaN,1,0,0,1,0,0,NaN,1.0,...,2.0,2.0,2.0,-1616.0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,2.0
8,152670,0,Cash loans,F,N,N,0,81000.0,45000.0,2187.0,45000.0,Unaccompanied,Commercial associate,Secondary / secondary special,Civil marriage,Municipal apartment,0.026392,-13643,-2138,-6118.0,-4127,NaN,1,1,0,1,0,1,Laborers,2.0,...,0.0,3.0,0.0,-393.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
9,206992,0,Cash loans,M,N,Y,2,180000.0,45000.0,2187.0,45000.0,Unaccompanied,Working,Higher education,Married,House / apartment,0.020246,-14143,-455,-390.0,-4770,NaN,1,1,1,1,0,0,NaN,4.0,...,0.0,0.0,0.0,-1530.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0


**Your answer (3.1):**

**I 3.1:** Select distinct will show you all the unique values - so that works better for typos on strings. Honestly not sure for this kinda of value. I would either visualise to see outliers or just use quantiles to see them - both in python tho not SQL.

...



## 4 — CASE and derived columns

`CASE` is SQL's if/else. Use it to band a continuous variable, recode categories, or build a 0/1 flag. It appears in the `SELECT` list, and — powerfully — inside aggregates (Section 14).

**Key syntax**

```sql
CASE
    WHEN AGE_YEARS < 30 THEN '<30'
    WHEN AGE_YEARS < 45 THEN '30-44'
    WHEN AGE_YEARS < 60 THEN '45-59'
    ELSE '60+'
END AS age_band,

CASE WHEN AMT_INCOME_TOTAL > 200000 THEN 1 ELSE 0 END AS high_income_flag
```

**Worked example** — age band + a credit-burden flag:

In [ ]:
q('''
SELECT SK_ID_CURR,
       -DAYS_BIRTH / 365.25 AS age_years,
       CASE
           WHEN -DAYS_BIRTH / 365.25 < 30 THEN '<30'
           WHEN -DAYS_BIRTH / 365.25 < 45 THEN '30-44'
           WHEN -DAYS_BIRTH / 365.25 < 60 THEN '45-59'
           ELSE '60+'
       END AS age_band,
       CASE WHEN AMT_CREDIT / AMT_INCOME_TOTAL > 4 THEN 1 ELSE 0 END AS high_burden
FROM   application_train
LIMIT  15
''')


### Exercises

**4.1** — Add a column `income_band` that buckets `AMT_INCOME_TOTAL` into `'<100k'`, `'100k-200k'`, `'200k-400k'`, `'400k+'`. Show it alongside `SK_ID_CURR` and `AMT_INCOME_TOTAL`, 20 rows.

**4.2** — Build a 0/1 `owns_both` flag = 1 when the client owns a car and realty, else 0. Show 20 rows.

**4.3** — Recode `CODE_GENDER` into a column `gender_clean` that maps `'XNA'` to `NULL` and leaves `'M'`/`'F'` as-is. (Hint: `CASE WHEN CODE_GENDER = 'XNA' THEN NULL ELSE CODE_GENDER END`.)

**4.4** — Create a `family_size_band`: `'1'`, `'2'`, `'3-4'`, `'5+'` from `CNT_FAM_MEMBERS`. Show 20 rows.

**Interpretation 4.1** — Banding a continuous variable loses information. Why do credit scorecards use bands anyway? Name one upside for a risk committee.

In [ ]:
# Ex 4.1
q('''
  SELECT 
    SK_ID_CURR,
    AMT_INCOME_TOTAL,
    CASE
        WHEN AMT_INCOME_TOTAL < 100000 THEN '<$100k'
        WHEN AMT_INCOME_TOTAL < 200000 THEN '$100k - $200k'
        WHEN AMT_INCOME_TOTAL < 400000 THEN '$200k - $400k'
        ELSE '$400k+'
        END AS income_band
        
    FROM application_train
    LIMIT 20

''')


In [46]:
# Ex 4.2
q('''
SELECT *,
    CASE
        WHEN FLAG_OWN_CAR = 'Y' AND FLAG_OWN_REALTY = 'Y' THEN 1
        ELSE 0
    END AS car_house_own_flag
FROM application_train
LIMIT 20
''')

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,NAME_TYPE_SUITE,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,REGION_POPULATION_RELATIVE,DAYS_BIRTH,DAYS_EMPLOYED,DAYS_REGISTRATION,DAYS_ID_PUBLISH,OWN_CAR_AGE,FLAG_MOBIL,FLAG_EMP_PHONE,FLAG_WORK_PHONE,FLAG_CONT_MOBILE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,...,OBS_60_CNT_SOCIAL_CIRCLE,DEF_60_CNT_SOCIAL_CIRCLE,DAYS_LAST_PHONE_CHANGE,FLAG_DOCUMENT_2,FLAG_DOCUMENT_3,FLAG_DOCUMENT_4,FLAG_DOCUMENT_5,FLAG_DOCUMENT_6,FLAG_DOCUMENT_7,FLAG_DOCUMENT_8,FLAG_DOCUMENT_9,FLAG_DOCUMENT_10,FLAG_DOCUMENT_11,FLAG_DOCUMENT_12,FLAG_DOCUMENT_13,FLAG_DOCUMENT_14,FLAG_DOCUMENT_15,FLAG_DOCUMENT_16,FLAG_DOCUMENT_17,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR,car_house_own_flag
0,100002,1,Cash loans,M,N,Y,0,202500.000,406597.5,24700.5,351000.0,Unaccompanied,Working,Secondary / secondary special,Single / not married,House / apartment,0.018801,-9461,-637,-3648.0,-2120,NaN,1,1,0,1,1,0,Laborers,1.0,...,2.0,2.0,-1134.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0,0
1,100003,0,Cash loans,F,N,N,0,270000.000,1293502.5,35698.5,1129500.0,Family,State servant,Higher education,Married,House / apartment,0.003541,-16765,-1188,-1186.0,-291,NaN,1,1,0,1,1,0,Core staff,2.0,...,1.0,0.0,-828.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0
2,100004,0,Revolving loans,M,Y,Y,0,67500.000,135000.0,6750.0,135000.0,Unaccompanied,Working,Secondary / secondary special,Single / not married,House / apartment,0.010032,-19046,-225,-4260.0,-2531,26.0,1,1,1,1,1,0,Laborers,1.0,...,0.0,0.0,-815.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,1
3,100006,0,Cash loans,F,N,Y,0,135000.000,312682.5,29686.5,297000.0,Unaccompanied,Working,Secondary / secondary special,Civil marriage,House / apartment,0.008019,-19005,-3039,-9833.0,-2437,NaN,1,1,0,1,0,0,Laborers,2.0,...,2.0,0.0,-617.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,0
4,100007,0,Cash loans,M,N,Y,0,121500.000,513000.0,21865.5,513000.0,Unaccompanied,Working,Secondary / secondary special,Single / not married,House / apartment,0.028663,-19932,-3038,-4311.0,-3458,NaN,1,1,0,1,0,0,Core staff,1.0,...,0.0,0.0,-1106.0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0
5,100008,0,Cash loans,M,N,Y,0,99000.000,490495.5,27517.5,454500.0,"Spouse, partner",State servant,Secondary / secondary special,Married,House / apartment,0.035792,-16941,-1588,-4970.0,-477,NaN,1,1,1,1,1,0,Laborers,2.0,...,0.0,0.0,-2536.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,1.0,1.0,0
6,100009,0,Cash loans,F,Y,Y,1,171000.000,1560726.0,41301.0,1395000.0,Unaccompanied,Commercial associate,Higher education,Married,House / apartment,0.035792,-13778,-3130,-1213.0,-619,17.0,1,1,0,1,1,0,Accountants,3.0,...,1.0,0.0,-1562.0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0.0,0.0,0.0,1.0,1.0,2.0,1
7,100010,0,Cash loans,M,Y,Y,0,360000.000,1530000.0,42075.0,1530000.0,Unaccompanied,State servant,Higher education,Married,House / apartment,0.003122,-18850,-449,-4597.0,-2379,8.0,1,1,1,1,0,0,Managers,2.0,...,2.0,0.0,-1070.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,1
8,100011,0,Cash loans,F,N,Y,0,112500.000,1019610.0,33826.5,913500.0,Children,Pensioner,Secondary / secondary special,Married,House / apartment,0.018634,-20099,365243,-7427.0,-3514,NaN,1,0,0,1,0,0,NaN,2.0,...,1.0,0.0,0.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0,0
9,100012,0,Revolving loans,M,N,Y,0,135000.000,405000.0,20250.0,405000.0,Unaccompanied,Working,Secondary / secondary special,Single / not married,House / apartment,0.019689,-14469,-2019,-14437.0,-3992,NaN,1,1,0,1,0,0,Laborers,1.0,...,2.0,0.0,-1673.0,0,0,0,0,0,0,0,0,0,0

In [57]:
# Ex 4.3
q('''
  SELECT *,
  CASE
    WHEN CODE_GENDER = 'XNA' THEN NULL ELSE CODE_GENDER END AS gender_clean
FROM application_train
LIMIT 20

''')


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,NAME_TYPE_SUITE,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,REGION_POPULATION_RELATIVE,DAYS_BIRTH,DAYS_EMPLOYED,DAYS_REGISTRATION,DAYS_ID_PUBLISH,OWN_CAR_AGE,FLAG_MOBIL,FLAG_EMP_PHONE,FLAG_WORK_PHONE,FLAG_CONT_MOBILE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,...,OBS_60_CNT_SOCIAL_CIRCLE,DEF_60_CNT_SOCIAL_CIRCLE,DAYS_LAST_PHONE_CHANGE,FLAG_DOCUMENT_2,FLAG_DOCUMENT_3,FLAG_DOCUMENT_4,FLAG_DOCUMENT_5,FLAG_DOCUMENT_6,FLAG_DOCUMENT_7,FLAG_DOCUMENT_8,FLAG_DOCUMENT_9,FLAG_DOCUMENT_10,FLAG_DOCUMENT_11,FLAG_DOCUMENT_12,FLAG_DOCUMENT_13,FLAG_DOCUMENT_14,FLAG_DOCUMENT_15,FLAG_DOCUMENT_16,FLAG_DOCUMENT_17,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR,gender_clean
0,100002,1,Cash loans,M,N,Y,0,202500.000,406597.5,24700.5,351000.0,Unaccompanied,Working,Secondary / secondary special,Single / not married,House / apartment,0.018801,-9461,-637,-3648.0,-2120,NaN,1,1,0,1,1,0,Laborers,1.0,...,2.0,2.0,-1134.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0,M
1,100003,0,Cash loans,F,N,N,0,270000.000,1293502.5,35698.5,1129500.0,Family,State servant,Higher education,Married,House / apartment,0.003541,-16765,-1188,-1186.0,-291,NaN,1,1,0,1,1,0,Core staff,2.0,...,1.0,0.0,-828.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,F
2,100004,0,Revolving loans,M,Y,Y,0,67500.000,135000.0,6750.0,135000.0,Unaccompanied,Working,Secondary / secondary special,Single / not married,House / apartment,0.010032,-19046,-225,-4260.0,-2531,26.0,1,1,1,1,1,0,Laborers,1.0,...,0.0,0.0,-815.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,M
3,100006,0,Cash loans,F,N,Y,0,135000.000,312682.5,29686.5,297000.0,Unaccompanied,Working,Secondary / secondary special,Civil marriage,House / apartment,0.008019,-19005,-3039,-9833.0,-2437,NaN,1,1,0,1,0,0,Laborers,2.0,...,2.0,0.0,-617.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,F
4,100007,0,Cash loans,M,N,Y,0,121500.000,513000.0,21865.5,513000.0,Unaccompanied,Working,Secondary / secondary special,Single / not married,House / apartment,0.028663,-19932,-3038,-4311.0,-3458,NaN,1,1,0,1,0,0,Core staff,1.0,...,0.0,0.0,-1106.0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,M
5,100008,0,Cash loans,M,N,Y,0,99000.000,490495.5,27517.5,454500.0,"Spouse, partner",State servant,Secondary / secondary special,Married,House / apartment,0.035792,-16941,-1588,-4970.0,-477,NaN,1,1,1,1,1,0,Laborers,2.0,...,0.0,0.0,-2536.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,1.0,1.0,M
6,100009,0,Cash loans,F,Y,Y,1,171000.000,1560726.0,41301.0,1395000.0,Unaccompanied,Commercial associate,Higher education,Married,House / apartment,0.035792,-13778,-3130,-1213.0,-619,17.0,1,1,0,1,1,0,Accountants,3.0,...,1.0,0.0,-1562.0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0.0,0.0,0.0,1.0,1.0,2.0,F
7,100010,0,Cash loans,M,Y,Y,0,360000.000,1530000.0,42075.0,1530000.0,Unaccompanied,State servant,Higher education,Married,House / apartment,0.003122,-18850,-449,-4597.0,-2379,8.0,1,1,1,1,0,0,Managers,2.0,...,2.0,0.0,-1070.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,M
8,100011,0,Cash loans,F,N,Y,0,112500.000,1019610.0,33826.5,913500.0,Children,Pensioner,Secondary / secondary special,Married,House / apartment,0.018634,-20099,365243,-7427.0,-3514,NaN,1,0,0,1,0,0,NaN,2.0,...,1.0,0.0,0.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0,F
9,100012,0,Revolving loans,M,N,Y,0,135000.000,405000.0,20250.0,405000.0,Unaccompanied,Working,Secondary / secondary special,Single / not married,House / apartment,0.019689,-14469,-2019,-14437.0,-3992,NaN,1,1,0,1,0,0,Laborers,1.0,...,2.0,0.0,-1673.0,0,0,0,0,0,0,0,0,0,0,0,0,0

In [54]:
# Ex 4.4
q('''
  SELECT *,
    CASE
        WHEN CNT_FAM_MEMBERS = 1 THEN '1'
        WHEN CNT_FAM_MEMBERS = 2 THEN '2'
        WHEN CNT_FAM_MEMBERS >= 3 AND CNT_FAM_MEMBERS <= 4 THEN '3 - 4'
        ELSE '5+'
    END AS famiy_size_band
    FROM application_train
    LIMIT 20
        

''')


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,NAME_TYPE_SUITE,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,REGION_POPULATION_RELATIVE,DAYS_BIRTH,DAYS_EMPLOYED,DAYS_REGISTRATION,DAYS_ID_PUBLISH,OWN_CAR_AGE,FLAG_MOBIL,FLAG_EMP_PHONE,FLAG_WORK_PHONE,FLAG_CONT_MOBILE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,...,OBS_60_CNT_SOCIAL_CIRCLE,DEF_60_CNT_SOCIAL_CIRCLE,DAYS_LAST_PHONE_CHANGE,FLAG_DOCUMENT_2,FLAG_DOCUMENT_3,FLAG_DOCUMENT_4,FLAG_DOCUMENT_5,FLAG_DOCUMENT_6,FLAG_DOCUMENT_7,FLAG_DOCUMENT_8,FLAG_DOCUMENT_9,FLAG_DOCUMENT_10,FLAG_DOCUMENT_11,FLAG_DOCUMENT_12,FLAG_DOCUMENT_13,FLAG_DOCUMENT_14,FLAG_DOCUMENT_15,FLAG_DOCUMENT_16,FLAG_DOCUMENT_17,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR,famiy_size_band
0,100002,1,Cash loans,M,N,Y,0,202500.000,406597.5,24700.5,351000.0,Unaccompanied,Working,Secondary / secondary special,Single / not married,House / apartment,0.018801,-9461,-637,-3648.0,-2120,NaN,1,1,0,1,1,0,Laborers,1.0,...,2.0,2.0,-1134.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0,1
1,100003,0,Cash loans,F,N,N,0,270000.000,1293502.5,35698.5,1129500.0,Family,State servant,Higher education,Married,House / apartment,0.003541,-16765,-1188,-1186.0,-291,NaN,1,1,0,1,1,0,Core staff,2.0,...,1.0,0.0,-828.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,2
2,100004,0,Revolving loans,M,Y,Y,0,67500.000,135000.0,6750.0,135000.0,Unaccompanied,Working,Secondary / secondary special,Single / not married,House / apartment,0.010032,-19046,-225,-4260.0,-2531,26.0,1,1,1,1,1,0,Laborers,1.0,...,0.0,0.0,-815.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,1
3,100006,0,Cash loans,F,N,Y,0,135000.000,312682.5,29686.5,297000.0,Unaccompanied,Working,Secondary / secondary special,Civil marriage,House / apartment,0.008019,-19005,-3039,-9833.0,-2437,NaN,1,1,0,1,0,0,Laborers,2.0,...,2.0,0.0,-617.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,2
4,100007,0,Cash loans,M,N,Y,0,121500.000,513000.0,21865.5,513000.0,Unaccompanied,Working,Secondary / secondary special,Single / not married,House / apartment,0.028663,-19932,-3038,-4311.0,-3458,NaN,1,1,0,1,0,0,Core staff,1.0,...,0.0,0.0,-1106.0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,1
5,100008,0,Cash loans,M,N,Y,0,99000.000,490495.5,27517.5,454500.0,"Spouse, partner",State servant,Secondary / secondary special,Married,House / apartment,0.035792,-16941,-1588,-4970.0,-477,NaN,1,1,1,1,1,0,Laborers,2.0,...,0.0,0.0,-2536.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,1.0,1.0,2
6,100009,0,Cash loans,F,Y,Y,1,171000.000,1560726.0,41301.0,1395000.0,Unaccompanied,Commercial associate,Higher education,Married,House / apartment,0.035792,-13778,-3130,-1213.0,-619,17.0,1,1,0,1,1,0,Accountants,3.0,...,1.0,0.0,-1562.0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0.0,0.0,0.0,1.0,1.0,2.0,3 - 4
7,100010,0,Cash loans,M,Y,Y,0,360000.000,1530000.0,42075.0,1530000.0,Unaccompanied,State servant,Higher education,Married,House / apartment,0.003122,-18850,-449,-4597.0,-2379,8.0,1,1,1,1,0,0,Managers,2.0,...,2.0,0.0,-1070.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,2
8,100011,0,Cash loans,F,N,Y,0,112500.000,1019610.0,33826.5,913500.0,Children,Pensioner,Secondary / secondary special,Married,House / apartment,0.018634,-20099,365243,-7427.0,-3514,NaN,1,0,0,1,0,0,NaN,2.0,...,1.0,0.0,0.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0,2
9,100012,0,Revolving loans,M,N,Y,0,135000.000,405000.0,20250.0,405000.0,Unaccompanied,Working,Secondary / secondary special,Single / not married,House / apartment,0.019689,-14469,-2019,-14437.0,-3992,NaN,1,1,0,1,0,0,Laborers,1.0,...,2.0,0.0,-1673.0,0,0,0,0,0,0,0,0,0,

**Your answer (4.1):**

**I 4.1:** Banding allows for segmentation and to narrow down risk to specific areas/flags/demographics - communicating these segments without banding is very difficult.

...

## 5 — Aggregate functions

Aggregates collapse many rows into one number: `COUNT`, `SUM`, `AVG`, `MIN`, `MAX`, `MEDIAN`, `STDDEV`. `COUNT(*)` counts rows; `COUNT(col)` counts non-NULLs; `COUNT(DISTINCT col)` counts unique values.

**The credit-risk trick:** because `TARGET` is 0/1, `AVG(TARGET)` **is the default rate**, and `SUM(TARGET)` is the number of defaulters. You will use `AVG(TARGET)` in almost every analytical query.

**Key syntax**

```sql
SELECT COUNT(*)                         AS n_rows,
       COUNT(OCCUPATION_TYPE)           AS n_with_occupation,
       COUNT(DISTINCT NAME_INCOME_TYPE) AS n_income_types,
       AVG(TARGET)                      AS default_rate,
       SUM(TARGET)                      AS n_defaults,
       MEDIAN(AMT_INCOME_TOTAL)         AS median_income
FROM   application_train;
```

**Worked example** — portfolio headline numbers:

In [ ]:
q('''
SELECT COUNT(*)                 AS n_applications,
       SUM(TARGET)              AS n_defaults,
       AVG(TARGET)              AS default_rate,
       AVG(AMT_CREDIT)          AS mean_credit,
       MEDIAN(AMT_INCOME_TOTAL) AS median_income,
       MAX(AMT_INCOME_TOTAL)    AS max_income
FROM   application_train
''')


### Exercises

**5.1** — In one query, return the total number of applications, the number of defaulters, and the overall default rate. Remember this rate — it is your portfolio baseline for the rest of the notebook.

**5.2** — What fraction of applications have a non-NULL `EXT_SOURCE_1`? (Hint: `COUNT(EXT_SOURCE_1) * 1.0 / COUNT(*)`.)

**5.3** — Return `MIN`, `MAX`, `AVG`, and `MEDIAN` of `age_years` (`-DAYS_BIRTH/365.25`).

**5.4** — Count the distinct number of `ORGANIZATION_TYPE` values.

**Interpretation 5.1** — Your baseline default rate is about 8%. Why does that single number make "accuracy" a misleading metric for a default model? (Think about a model that predicts "no default" for everyone.)

In [ ]:
# Ex 5.1
q('''

''')


In [ ]:
# Ex 5.2
q('''

''')


In [ ]:
# Ex 5.3
q('''

''')


In [ ]:
# Ex 5.4
q('''

''')


**Your answer (5.1):**

...

## 6 — GROUP BY and HAVING

`GROUP BY` splits rows into groups and runs the aggregates *per group* — this is the workhorse of analyst SQL. `HAVING` filters **groups** after aggregation (whereas `WHERE` filters rows before it).

**Key syntax**

```sql
SELECT   NAME_EDUCATION_TYPE,
         COUNT(*)    AS n,
         AVG(TARGET) AS default_rate
FROM     application_train
GROUP BY NAME_EDUCATION_TYPE
HAVING   COUNT(*) > 1000          -- drop tiny, noisy groups
ORDER BY default_rate DESC;
```

Rule of thumb: every non-aggregated column in `SELECT` must appear in `GROUP BY`. `WHERE` cannot see aggregates; `HAVING` can.

**Worked example** — default rate by education, biggest risk first:

In [ ]:
q('''
SELECT   NAME_EDUCATION_TYPE,
         COUNT(*)    AS n,
         AVG(TARGET) AS default_rate
FROM     application_train
GROUP BY NAME_EDUCATION_TYPE
ORDER BY default_rate DESC
''')


### Exercises

**6.1** — Default rate and count by `NAME_INCOME_TYPE`, sorted by default rate descending.

**6.2** — Default rate by `age_band` (reuse your Section 4 CASE). Order by the band, not the rate, so it reads young→old.

**6.3** — Default rate by `OCCUPATION_TYPE`, but only for occupations with at least 5,000 applications (use `HAVING`). Sort descending.

**6.4** — Default rate by the pair `(CODE_GENDER, NAME_EDUCATION_TYPE)`. Drop the `XNA` gender in a `WHERE`. Sort by rate descending.

**6.5** — For each `NAME_FAMILY_STATUS`, return count, default rate, and average `AMT_CREDIT`.

**Interpretation 6.1** — From 6.3: name the two highest-risk and two lowest-risk occupations. What underlying factor (income stability? seasonality?) plausibly links the risky ones?
**Interpretation 6.2** — Why did we add `HAVING COUNT(*) > 5000`? What goes wrong if you rank occupations by default rate without a size filter?

In [ ]:
# Ex 6.1
q('''

''')


In [ ]:
# Ex 6.2
q('''

''')


In [ ]:
# Ex 6.3
q('''

''')


In [ ]:
# Ex 6.4
q('''

''')


In [ ]:
# Ex 6.5
q('''

''')


**Your answer (6.1, 6.2):**

...

## 7 — NULL handling and the DAYS_EMPLOYED sentinel

`NULL` means "unknown", and it is contagious: `1 + NULL = NULL`, and aggregates *skip* NULLs (`AVG` ignores them, `COUNT(col)` doesn't count them). Tools you need:

- `COALESCE(a, b, c)` — first non-NULL argument (great for fallbacks/imputation).
- `NULLIF(a, b)` — returns NULL when `a = b` (great for turning a **sentinel** into a real NULL, and for guarding divide-by-zero).
- `TRY_CAST(x AS type)` — cast, returning NULL instead of erroring.

**The real-world catch in this data:** `DAYS_EMPLOYED` has a sentinel `365243` (~1000 years) standing in for "not employed / not applicable". Left alone it poisons every average. Convert it to NULL with `NULLIF`.

**Worked example** — see the sentinel, then neutralise it:

In [ ]:
q('''
SELECT
    MAX(DAYS_EMPLOYED)                              AS raw_max,      -- 365243 sentinel
    AVG(-DAYS_EMPLOYED / 365.25)                    AS dirty_avg_years,
    AVG(-NULLIF(DAYS_EMPLOYED, 365243) / 365.25)    AS clean_avg_years,
    COUNT(*) - COUNT(NULLIF(DAYS_EMPLOYED, 365243)) AS n_sentinel
FROM application_train
''')


### Exercises

**7.1** — Count how many rows have the `DAYS_EMPLOYED` sentinel (`= 365243`). What fraction of the portfolio is that?

**7.2** — Compute `COALESCE(AMT_ANNUITY, 0)` for the rows where `AMT_ANNUITY IS NULL` — show that the coalesced value is 0. (Filter to `AMT_ANNUITY IS NULL`, show 10 rows with both raw and coalesced.)

**7.3** — Safe ratio: compute `AMT_CREDIT / NULLIF(AMT_GOODS_PRICE, 0)` so a zero goods price yields NULL instead of an error. Show 10 rows.

**7.4** — For each `NAME_INCOME_TYPE`, what fraction of rows have a **missing** `EXT_SOURCE_1`? (Hint: `AVG(CASE WHEN EXT_SOURCE_1 IS NULL THEN 1 ELSE 0 END)`.)

**Interpretation 7.1** — Why is turning the sentinel into NULL *better* than deleting those rows? What information would deletion throw away, and how might you preserve it as a feature?
**Interpretation 7.2** — When is `COALESCE(x, 0)` a mistake? Give a column in this data where imputing 0 would be wrong.

In [ ]:
# Ex 7.1
q('''

''')


In [ ]:
# Ex 7.2
q('''

''')


In [ ]:
# Ex 7.3
q('''

''')


In [ ]:
# Ex 7.4
q('''

''')


**Your answer (7.1, 7.2):**

...

## 8 — Joins I: combining tables

A join stitches rows from two tables on a shared key. The types you need:

- `INNER JOIN` — only rows matching in both tables.
- `LEFT JOIN` — every row of the left table, with NULLs where the right has no match. **This is the default for analyst work**: you want to keep every application, even clients with no bureau history.

**Key syntax**

```sql
SELECT a.SK_ID_CURR, a.TARGET, b.CREDIT_TYPE
FROM   application_train a
LEFT JOIN bureau b ON a.SK_ID_CURR = b.SK_ID_CURR
```

**The fan-out warning (grain again):** `bureau` has many rows per client, so this join **multiplies** application rows. That is fine for inspecting, but you must never aggregate application-level columns on top of a fanned-out join. The fix is Section 9.

**Worked example** — one client's applications joined to their bureau records (watch the client repeat):

In [ ]:
q('''
SELECT a.SK_ID_CURR, a.TARGET, b.SK_ID_BUREAU, b.CREDIT_ACTIVE, b.AMT_CREDIT_SUM
FROM   application_train a
LEFT JOIN bureau b ON a.SK_ID_CURR = b.SK_ID_CURR
WHERE  a.SK_ID_CURR = 100002
''')


### Exercises

**8.1** — `INNER JOIN` `application_train` to `bureau`. Count the resulting rows. Then `LEFT JOIN` and count again. Why is the LEFT result larger?

**8.2** — How many *distinct* clients in `application_train` have **at least one** bureau record? (INNER JOIN, then `COUNT(DISTINCT a.SK_ID_CURR)`.)

**8.3** — How many clients have **no** bureau record at all? (LEFT JOIN and count where `b.SK_ID_BUREAU IS NULL`.)

**8.4** — For client `SK_ID_CURR = 100002`, list all their `previous_application` rows: `SK_ID_PREV`, `NAME_CONTRACT_STATUS`, `AMT_APPLICATION`, `AMT_CREDIT`.

**Interpretation 8.1** — A client with no bureau history joins to all-NULLs on a LEFT JOIN. Is "no external credit history" more likely to be higher or lower risk here, and why can't you assume it means *safe*?
**Interpretation 8.2** — Explain, in grain terms, why you must not write `AVG(a.AMT_INCOME_TOTAL)` over the raw application-to-bureau join.

In [ ]:
# Ex 8.1
q('''

''')


In [ ]:
# Ex 8.2
q('''

''')


In [ ]:
# Ex 8.3
q('''

''')


In [ ]:
# Ex 8.4
q('''

''')


**Your answer (8.1, 8.2):**

...

## 9 — The aggregate-then-join pattern (the most important one)

This is *the* pattern for building client-level features from child tables, and the backbone of the capstone. Steps:

1. Aggregate the child table down to **one row per client** in a subquery / CTE.
2. `LEFT JOIN` that summary back onto `application_train`.
3. Now application-level aggregates are safe again.

**Key syntax**

```sql
WITH bureau_agg AS (
    SELECT SK_ID_CURR,
           COUNT(*)                 AS n_bureau,
           SUM(AMT_CREDIT_SUM_DEBT) AS total_debt
    FROM   bureau
    GROUP BY SK_ID_CURR
)
SELECT a.SK_ID_CURR, a.TARGET, ba.n_bureau, ba.total_debt
FROM   application_train a
LEFT JOIN bureau_agg ba ON a.SK_ID_CURR = ba.SK_ID_CURR;
```

**Worked example** — does *number of prior bureau credits* relate to default? Aggregate first, band the count, then group:

In [ ]:
q('''
WITH bureau_agg AS (
    SELECT SK_ID_CURR, COUNT(*) AS n_bureau
    FROM   bureau
    GROUP BY SK_ID_CURR
)
SELECT
    CASE
        WHEN ba.n_bureau IS NULL THEN '0 (no history)'
        WHEN ba.n_bureau <= 2    THEN '1-2'
        WHEN ba.n_bureau <= 5    THEN '3-5'
        ELSE '6+'
    END                          AS bureau_count_band,
    COUNT(*)                     AS n_clients,
    AVG(a.TARGET)                AS default_rate
FROM   application_train a
LEFT JOIN bureau_agg ba ON a.SK_ID_CURR = ba.SK_ID_CURR
GROUP BY 1
ORDER BY default_rate DESC
''')


### Exercises

**9.1** — Aggregate `bureau` to per-client `total_debt = SUM(AMT_CREDIT_SUM_DEBT)` and `n_active = SUM(CASE WHEN CREDIT_ACTIVE = 'Active' THEN 1 ELSE 0 END)`. LEFT JOIN onto applications and show 20 rows including `TARGET`.

**9.2** — Using your 9.1 aggregate, band `n_active` into `0 / 1-2 / 3+` (treat NULL as 0) and report default rate per band.

**9.3** — Aggregate `previous_application` to per-client `n_prev = COUNT(*)` and `n_refused = SUM(CASE WHEN NAME_CONTRACT_STATUS = 'Refused' THEN 1 ELSE 0 END)`. Join, then report default rate for clients who have **at least one** prior refusal vs those with none.

**9.4** — Compute each client's **refusal ratio** `n_refused / NULLIF(n_prev, 0)` from 9.3, band it (0, 0–0.5, 0.5+), and report default rate per band.

**Interpretation 9.1** — From 9.2/9.3: does having more active external credits, or a history of refusals, line up with higher default? Give a one-line risk narrative for each.
**Interpretation 9.2** — Why does `LEFT JOIN` + treat-NULL-as-0 matter here? What silently happens to no-history clients if you use `INNER JOIN` instead?

In [ ]:
# Ex 9.1
q('''

''')


In [ ]:
# Ex 9.2
q('''

''')


In [ ]:
# Ex 9.3
q('''

''')


In [ ]:
# Ex 9.4
q('''

''')


**Your answer (9.1, 9.2):**

...

## 10 — Multiple joins and chaining

Real feature tables pull from several children at once. Aggregate each to per-client grain in its own CTE, then LEFT JOIN them one after another onto the applications. Because each CTE is already one-row-per-client, there is no fan-out.

**Key syntax**

```sql
WITH b AS (SELECT SK_ID_CURR, ... FROM bureau GROUP BY SK_ID_CURR),
     p AS (SELECT SK_ID_CURR, ... FROM previous_application GROUP BY SK_ID_CURR)
SELECT a.SK_ID_CURR, a.TARGET, b.<...>, p.<...>
FROM   application_train a
LEFT JOIN b ON a.SK_ID_CURR = b.SK_ID_CURR
LEFT JOIN p ON a.SK_ID_CURR = p.SK_ID_CURR;
```

**Worked example** — combine bureau debt with prior-application activity in one query:

In [ ]:
q('''
WITH b AS (
    SELECT SK_ID_CURR, COUNT(*) AS n_bureau, SUM(AMT_CREDIT_SUM_DEBT) AS total_debt
    FROM bureau GROUP BY SK_ID_CURR
),
p AS (
    SELECT SK_ID_CURR, COUNT(*) AS n_prev,
           AVG(AMT_APPLICATION) AS avg_prev_app
    FROM previous_application GROUP BY SK_ID_CURR
)
SELECT a.SK_ID_CURR, a.TARGET,
       b.n_bureau, b.total_debt,
       p.n_prev, p.avg_prev_app
FROM application_train a
LEFT JOIN b ON a.SK_ID_CURR = b.SK_ID_CURR
LEFT JOIN p ON a.SK_ID_CURR = p.SK_ID_CURR
WHERE a.SK_ID_CURR IN (100002, 100003, 100004)
''')


### Exercises

**10.1** — Build one query that returns, per client: `n_bureau` (from bureau), `n_prev` (from previous_application), and `n_installments` (from installments_payments). LEFT JOIN all three. Show 20 rows with `TARGET`.

**10.2** — Add an installments feature: per client, `avg_payment_gap = AVG(AMT_INSTALMENT - AMT_PAYMENT)` (positive = underpaid). Join it on and show 20 rows.

**10.3** — Combine: for clients with at least 1 prior refusal (from `previous_application`) **and** positive `total_debt` (from `bureau`), what is the default rate vs everyone else?

**Interpretation 10.1** — Each LEFT JOIN adds NULLs for clients missing from that child. When you later feed this to a model, why does *how* you fill those NULLs (0 vs median vs a "missing" flag) change the story a feature tells?

In [ ]:
# Ex 10.1
q('''

''')


In [ ]:
# Ex 10.2
q('''

''')


In [ ]:
# Ex 10.3
q('''

''')


**Your answer (10.1):**

...

## 11 — Subqueries and CTEs

A **subquery** is a query nested inside another. A **CTE** (`WITH name AS (...)`) is a named subquery that reads top-to-bottom — prefer CTEs for anything non-trivial; they are far more readable and can be reused.

Flavours:
- **Scalar subquery** — returns one value, usable inline: `WHERE AMT_INCOME_TOTAL > (SELECT AVG(AMT_INCOME_TOTAL) FROM application_train)`.
- **`IN` / `NOT IN`** — membership against a column of values.
- **`EXISTS`** — true if the correlated subquery returns any row (often faster than `IN`).

**Key syntax**

```sql
SELECT *
FROM   application_train a
WHERE  EXISTS (SELECT 1 FROM bureau b
               WHERE b.SK_ID_CURR = a.SK_ID_CURR
                 AND b.CREDIT_ACTIVE = 'Active');
```

**Worked example** — above-average-income clients, with the portfolio average shown for context:

In [ ]:
q('''
WITH stats AS (SELECT AVG(AMT_INCOME_TOTAL) AS avg_income FROM application_train)
SELECT COUNT(*) AS n_above_avg,
       (SELECT avg_income FROM stats) AS portfolio_avg_income,
       AVG(a.TARGET) AS default_rate_above_avg
FROM application_train a, stats
WHERE a.AMT_INCOME_TOTAL > stats.avg_income
''')


### Exercises

**11.1** — Return the count of applications whose `AMT_CREDIT` is above the overall average `AMT_CREDIT` (use a scalar subquery).

**11.2** — Using `EXISTS`, count clients who have at least one **Active** bureau credit. Then count those who have **none** (`NOT EXISTS`).

**11.3** — Rewrite the Section 9 "prior refusal" analysis with `EXISTS`: default rate for clients where a refused `previous_application` exists vs where it does not.

**11.4** — With a CTE, compute per-education-type default rates, then in the outer query keep only education types whose rate is **above the portfolio average** (scalar subquery for the baseline).

**Interpretation 11.1** — When would you reach for a CTE over a nested subquery, purely for the next analyst who reads your code?
**Interpretation 11.2** — `NOT IN` has a famous NULL trap: if the subquery returns any NULL, `NOT IN` returns no rows. Why does `NOT EXISTS` avoid this?

In [ ]:
# Ex 11.1
q('''

''')


In [ ]:
# Ex 11.2
q('''

''')


In [ ]:
# Ex 11.3
q('''

''')


In [ ]:
# Ex 11.4
q('''

''')


**Your answer (11.1, 11.2):**

...

## 12 — Window functions I: ranking and deciles

A window function computes across a set of rows **related to the current row** without collapsing them — you keep every row *and* get an aggregate/rank alongside. The engine of scorecard analytics.

**Key syntax**

```sql
func() OVER (PARTITION BY group_col ORDER BY sort_col)
```

- `ROW_NUMBER()` — 1,2,3… unique.
- `RANK()` / `DENSE_RANK()` — ties share a rank.
- `NTILE(10)` — split rows into 10 equal buckets → **deciles**, exactly how you'd bucket a credit score.
- `AVG(x) OVER (PARTITION BY g)` — the group average attached to every row (no GROUP BY, no join).

**Worked example** — decile clients by `EXT_SOURCE_2` (an external score), then measure default rate per decile. This is the single most important query shape in scorecard validation — a good score shows a clean monotonic gradient:

In [ ]:
q('''
WITH scored AS (
    SELECT SK_ID_CURR, TARGET,
           NTILE(10) OVER (ORDER BY EXT_SOURCE_2) AS score_decile
    FROM   application_train
    WHERE  EXT_SOURCE_2 IS NOT NULL
)
SELECT score_decile,
       COUNT(*)    AS n,
       AVG(TARGET) AS default_rate
FROM   scored
GROUP BY score_decile
ORDER BY score_decile
''')


### Exercises

**12.1** — Repeat the decile analysis for `EXT_SOURCE_3`. Does default rate fall monotonically as the score decile rises?

**12.2** — Use `ROW_NUMBER() OVER (PARTITION BY NAME_INCOME_TYPE ORDER BY AMT_INCOME_TOTAL DESC)` to find the **top-3 highest earners within each** `NAME_INCOME_TYPE`. Keep only rows where that number ≤ 3.

**12.3** — Attach each client's `AVG(AMT_CREDIT) OVER (PARTITION BY NAME_EDUCATION_TYPE)` next to their own `AMT_CREDIT`, and compute the difference (how far above/below their education-group mean they borrowed). Show 20 rows.

**12.4** — Decile clients by `AMT_CREDIT / AMT_INCOME_TOTAL` (credit burden) and report default rate per decile. Filter out NULL/zero income first.

**Interpretation 12.1** — A well-behaved score gives monotonically falling default rates across deciles. From 12.1, does `EXT_SOURCE_3` behave like a good score? What would a *non*-monotonic pattern warn you about?
**Interpretation 12.2** — `RANK` vs `ROW_NUMBER` vs `DENSE_RANK`: describe how each treats a tie, and when picking the wrong one changes your "top N per group" answer.

In [ ]:
# Ex 12.1
q('''

''')


In [ ]:
# Ex 12.2
q('''

''')


In [ ]:
# Ex 12.3
q('''

''')


In [ ]:
# Ex 12.4
q('''

''')


**Your answer (12.1, 12.2):**

...

## 13 — Window functions II: time series with LAG, running totals

Add an `ORDER BY` inside the window and you get **ordered** calculations: running totals, moving averages, and row-to-row comparisons with `LAG`/`LEAD`. Perfect for the monthly/installment child tables.

**Key syntax**

```sql
SUM(x)  OVER (PARTITION BY id ORDER BY t
              ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)  AS running_total,
AVG(x)  OVER (PARTITION BY id ORDER BY t
              ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)          AS moving_avg_3,
LAG(x)  OVER (PARTITION BY id ORDER BY t)                        AS prev_value
```

**Worked example** — one prior loan's installments over time: the payment, the previous payment (`LAG`), and a running total of amounts paid:

In [ ]:
q('''
WITH one_loan AS (SELECT SK_ID_PREV FROM installments_payments LIMIT 1)
SELECT SK_ID_PREV, NUM_INSTALMENT_NUMBER,
       DAYS_INSTALMENT,
       AMT_PAYMENT,
       LAG(AMT_PAYMENT) OVER (PARTITION BY SK_ID_PREV ORDER BY NUM_INSTALMENT_NUMBER) AS prev_payment,
       SUM(AMT_PAYMENT) OVER (PARTITION BY SK_ID_PREV ORDER BY NUM_INSTALMENT_NUMBER
                              ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)        AS running_paid
FROM   installments_payments
WHERE  SK_ID_PREV = (SELECT SK_ID_PREV FROM one_loan)
ORDER BY NUM_INSTALMENT_NUMBER
LIMIT 20
''')


### Exercises

**13.1** — For a single `SK_ID_PREV` of your choice, show each installment with a `late_days = DAYS_ENTRY_PAYMENT - DAYS_INSTALMENT` column (positive = paid late) and a 3-installment moving average of `AMT_PAYMENT`.

**13.2** — Using `pos_cash_balance`, for one `SK_ID_PREV` order by `MONTHS_BALANCE` and show `SK_DPD` (days past due) with its `LAG` — did delinquency rise month over month?

**13.3** — Per client (`SK_ID_CURR`) in `installments_payments`, count how many installments were paid late (`DAYS_ENTRY_PAYMENT > DAYS_INSTALMENT`). Do this as an aggregate (no window needed) and return the 10 clients with the most late payments.

**13.4** — Build a per-client feature `max_dpd` = the maximum `SK_DPD` ever seen in `pos_cash_balance`, and a `n_months` count. (Aggregate to per-client grain — this is a capstone building block.)

**Interpretation 13.1** — Late-payment history is one of the strongest default predictors in practice. Which single per-client feature from this section would you build first, and why?
**Interpretation 13.2** — Windowed features can leak the future. If a feature's `ORDER BY` includes data dated *after* the application, why is that a problem, and how does the sign of `DAYS_*` help you avoid it?

In [ ]:
# Ex 13.1
q('''

''')


In [ ]:
# Ex 13.2
q('''

''')


In [ ]:
# Ex 13.3
q('''

''')


In [ ]:
# Ex 13.4
q('''

''')


**Your answer (13.1, 13.2):**

...

## 14 — Conditional aggregation and pivoting

Put a `CASE` (or DuckDB's `FILTER`) *inside* an aggregate to count/sum a subset — this builds crosstabs and one-row-per-client flag features in a single pass.

**Key syntax**

```sql
SUM(CASE WHEN CREDIT_ACTIVE = 'Active' THEN 1 ELSE 0 END) AS n_active,
COUNT(*) FILTER (WHERE CREDIT_ACTIVE = 'Closed')          AS n_closed,   -- DuckDB/Postgres
AVG(TARGET) FILTER (WHERE CODE_GENDER = 'F')              AS female_rate
```

**Worked example** — default rate split by contract type across each income type, as a pivot in one query:

In [ ]:
q('''
SELECT NAME_INCOME_TYPE,
       AVG(TARGET) FILTER (WHERE NAME_CONTRACT_TYPE = 'Cash loans')      AS rate_cash,
       AVG(TARGET) FILTER (WHERE NAME_CONTRACT_TYPE = 'Revolving loans') AS rate_revolving,
       COUNT(*) AS n
FROM   application_train
GROUP BY NAME_INCOME_TYPE
ORDER BY n DESC
''')


### Exercises

**14.1** — Per client, aggregate `bureau` into `n_active` and `n_closed` using `SUM(CASE ...)` (or `COUNT(*) FILTER`). Show 20 rows.

**14.2** — Build a pivot: rows = `NAME_EDUCATION_TYPE`, columns = default rate for `CODE_GENDER = 'M'` vs `'F'`, plus a count. Which education level shows the widest gender gap?

**14.3** — Per client in `previous_application`, count approvals, refusals, and cancellations in one row (three `FILTER`/`CASE` aggregates keyed on `NAME_CONTRACT_STATUS`).

**14.4** — Portfolio pivot: for each `age_band`, show the default rate for `CODE_GENDER='M'` and for `'F'` side by side.

**Interpretation 14.1** — Conditional aggregation lets you make one wide feature row per client. Why is that shape (one row per client, many columns) exactly what a model wants, and how does it relate to the grain lesson from Section 0?

In [ ]:
# Ex 14.1
q('''

''')


In [ ]:
# Ex 14.2
q('''

''')


In [ ]:
# Ex 14.3
q('''

''')


In [ ]:
# Ex 14.4
q('''

''')


**Your answer (14.1):**

...

## 15 — Set operations

Stack the results of two queries (same columns, same order):

- `UNION` — combine and de-duplicate.
- `UNION ALL` — combine, keep duplicates (faster; use when you know rows are distinct).
- `EXCEPT` — rows in the first query not in the second.
- `INTERSECT` — rows in both.

**Key syntax**

```sql
SELECT SK_ID_CURR FROM application_train WHERE CODE_GENDER = 'F'
EXCEPT
SELECT SK_ID_CURR FROM application_train WHERE FLAG_OWN_CAR = 'Y';
```

**Worked example** — clients who appear in `previous_application` but have **no** `bureau` record:

In [ ]:
q('''
SELECT COUNT(*) AS n_clients FROM (
    SELECT DISTINCT SK_ID_CURR FROM previous_application
    EXCEPT
    SELECT DISTINCT SK_ID_CURR FROM bureau
)
''')


### Exercises

**15.1** — Count clients who have a `bureau` record **and** a `previous_application` record (`INTERSECT` on `SK_ID_CURR`).

**15.2** — Count clients in `application_train` who have **no** row in `installments_payments` (`EXCEPT`).

**15.3** — Build one result set labelling each `SK_ID_CURR` as `'has_bureau'` or `'no_bureau'` using two `SELECT`s and `UNION ALL` (add a literal string column in each). Then count each label.

**Interpretation 15.1** — When is `UNION ALL` the right call over `UNION`, and what's the performance cost of getting it wrong on a big table?

In [ ]:
# Ex 15.1
q('''

''')


In [ ]:
# Ex 15.2
q('''

''')


In [ ]:
# Ex 15.3
q('''

''')


**Your answer (15.1):**

...

## 16 — Putting it together: a risk-segmentation query

No new syntax — this section rehearses combining CTEs, window deciles, joins, and conditional aggregation into the kind of query an analyst actually ships.

**Worked example** — cross-tabulate default rate by **external-score quintile × credit-burden band**, the sort of two-way risk grid a credit committee reads:

In [ ]:
q('''
WITH base AS (
    SELECT SK_ID_CURR, TARGET,
           NTILE(5) OVER (ORDER BY EXT_SOURCE_2) AS score_quintile,
           CASE WHEN AMT_CREDIT / NULLIF(AMT_INCOME_TOTAL,0) > 4 THEN 'high_burden'
                ELSE 'normal_burden' END AS burden
    FROM   application_train
    WHERE  EXT_SOURCE_2 IS NOT NULL AND AMT_INCOME_TOTAL > 0
)
SELECT score_quintile,
       AVG(TARGET) FILTER (WHERE burden = 'high_burden')   AS rate_high_burden,
       AVG(TARGET) FILTER (WHERE burden = 'normal_burden') AS rate_normal_burden,
       COUNT(*) AS n
FROM base
GROUP BY score_quintile
ORDER BY score_quintile
''')


### Exercises

**16.1** — Rebuild the grid as **age band × score quintile**, cells = default rate. Which corner of the grid is riskiest?

**16.2** — Join in the bureau `total_debt` aggregate, decile clients by it, and show default rate per debt decile *within* the worst external-score quintile only.

**16.3** — Produce a single "watchlist" query: clients in the worst score quintile **and** high burden **and** with at least one prior refusal. Return their count and default rate, and compare to portfolio baseline.

**Interpretation 16.1** — Two-way risk grids can create tiny, unreliable cells. What minimum cell count would you insist on before quoting a cell's rate to a committee, and why?
**Interpretation 16.2** — Your watchlist segment has a much higher default rate than baseline. Does that make it a good *policy* cutoff? What business cost sits on the other side of rejecting everyone in it?

In [ ]:
# Ex 16.1
q('''

''')


In [ ]:
# Ex 16.2
q('''

''')


In [ ]:
# Ex 16.3
q('''

''')


**Your answer (16.1, 16.2):**

...

## 17 — Capstone: build a model-ready feature table

Everything above, combined into the artefact a credit-risk analyst actually hands off: **one row per `SK_ID_CURR`**, `TARGET` attached, and a wide set of engineered features drawn from every child table. This mirrors the pandas feature-engineering capstone exactly — build it in both and compare.

**The brief** — with a single `WITH` block of per-client aggregates LEFT JOINed onto `application_train`, produce a view `client_features` containing at least:

From **`application_train`** directly: `TARGET`, `age_years`, `years_employed` (sentinel → NULL), `credit_income_ratio`, `annuity_income_ratio`, `EXT_SOURCE_1/2/3`.

From **`bureau`** (per client): `n_bureau`, `n_active_bureau`, `total_bureau_debt`, `max_bureau_overdue`.

From **`previous_application`** (per client): `n_prev`, `n_refused`, `refusal_ratio`.

From **`installments_payments`** (per client): `n_installments`, `n_late_payments` (`DAYS_ENTRY_PAYMENT > DAYS_INSTALMENT`), `avg_payment_gap` (`AVG(AMT_INSTALMENT - AMT_PAYMENT)`).

From **`pos_cash_balance`** (per client): `max_dpd` (`MAX(SK_DPD)`).

Requirements: use CTEs (one per child), LEFT JOIN so no client is dropped, guard divides with `NULLIF`, neutralise the `DAYS_EMPLOYED` sentinel. Then run two validations: (a) row count of `client_features` equals row count of `application_train`; (b) decile the table by `EXT_SOURCE_2` and confirm the default-rate gradient survives in your assembled table.

**Then write the analysis** (markdown cell below): pick your 5 strongest features and, for each, one *observation → decision* line. Example shape: "`n_late_payments` rises sharply with default rate → strong candidate feature; watch for clients with 0 prior loans who get NULL, decide 0-fill vs missing-flag."

**A leakage caution to state explicitly:** every feature here must be knowable *at application time*. The `DAYS_*` signs help — anything dated after the decision would leak the outcome. Note in your write-up which columns you'd double-check for this.

In [ ]:
# Capstone: assemble client_features
# Build each child aggregate as a CTE, then LEFT JOIN onto application_train.
# Develop piece by piece in scratch cells, then assemble one CREATE VIEW.

q('''
CREATE OR REPLACE VIEW client_features AS
WITH bureau_agg AS (
    -- TODO: n_bureau, n_active_bureau, total_bureau_debt, max_bureau_overdue
    SELECT SK_ID_CURR
    FROM bureau
    GROUP BY SK_ID_CURR
)
-- TODO: prev_agg, inst_agg, pos_agg CTEs

SELECT a.SK_ID_CURR,
       a.TARGET
       -- TODO: application-level derived cols + joined aggregates
FROM application_train a
-- TODO: LEFT JOIN each aggregate
''')

# then:  q('SELECT COUNT(*) FROM client_features')


In [ ]:
# Validation (a): row count matches application_train
q('''

''')


In [ ]:
# Validation (b): default-rate gradient by EXT_SOURCE_2 decile, on client_features
q('''

''')


**Capstone decision statements (pick 5 features):**

Feature 1:

Feature 2:

Feature 3:

Feature 4:

Feature 5:

Leakage check — columns I'd double-check:

## 18 — Self-test: can you answer these cold?

Close the notes. If any are shaky, redo that section.

1. Grain: what does one row of `application_train` mean vs one row of `bureau`? Why does that dictate "aggregate before you join"?
2. `WHERE` vs `HAVING` — one sentence each, and which one can see `AVG(TARGET)`.
3. Why is `AVG(TARGET)` the default rate, and why does that make the aggregate-then-group pattern your bread and butter?
4. `INNER` vs `LEFT JOIN` — which is the analyst default and why?
5. `COALESCE` vs `NULLIF` — one use of each on this data (name the column).
6. The `DAYS_EMPLOYED` sentinel: value, how you'd detect it, what you do about it.
7. `NTILE(10)` — what it produces and how you'd use it to validate a credit score.
8. `ROW_NUMBER` vs `RANK` vs `DENSE_RANK` on ties.
9. `LAG` and a running `SUM OVER (... ROWS ...)` — one credit-risk feature each.
10. Feature leakage: what it is, and one column you'd audit before shipping the capstone table.

**When you're done:** bring the completed worksheet — queries and written answers — back for marking. It feeds straight into the pandas feature workbook, where you'll rebuild this same feature table with `groupby` + `merge`.